# RNA assembly summary

This notebook compares the length-filtered RNAviralSPAdes, MEGAHIT and Trinity assemblies selected for this run. The combined_derreplicated FASTA is a per-sample pool after exact sequence/reverse-complement collapse, before VirSorter screening; it is not the final viral vOTU catalogue.

All samples, selected assemblers and zero-contig assemblies are shown. The provenance plot describes the combination step, not an independent estimate of assembly quality.

In [ ]:
# Import the libraries used by the workflow assembly notebooks.
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
from Bio import SeqIO

sns.set_style("ticks", {"axes.grid": True})
sns.set_palette("colorblind")
plt.rcParams["axes.linewidth"] = 1.5
plt.rcParams["xtick.major.width"] = 1.5
plt.rcParams["ytick.major.width"] = 1.5
plt.rcParams["xtick.major.size"] = 8
plt.rcParams["ytick.major.size"] = 8
plt.rcParams["axes.titlepad"] = 20
plt.rcParams["axes.titlesize"] = 24
plt.rcParams["axes.labelsize"] = 17
plt.rcParams["xtick.labelsize"] = 12
plt.rcParams["ytick.labelsize"] = 12
plt.rcParams["legend.fontsize"] = 12
plt.rcParams["font.family"] = "sans-serif"
plt.rcParams["font.sans-serif"] = ["Liberation Sans"]
plt.rcParams["text.usetex"] = False
plt.rcParams["svg.fonttype"] = "none"
plt.rcParams["savefig.dpi"] = 300

## Input files and sample order

In [ ]:
SAMPLES = list(snakemake.params.samples)
ASSEMBLERS = list(snakemake.params.assemblers)
MIN_LENGTH = int(snakemake.params.min_length)
assembly_files = list(snakemake.input.assemblies)
combined_files = list(snakemake.input.combined)
provenance_files = list(snakemake.input.provenance)

if not SAMPLES or not ASSEMBLERS:
    raise ValueError("The RNA assembly report needs at least one sample and assembler")
if len(assembly_files) != len(SAMPLES) * len(ASSEMBLERS):
    raise ValueError("Expected one RNA assembly FASTA per sample and selected assembler")
if len(combined_files) != len(SAMPLES) or len(provenance_files) != len(SAMPLES):
    raise ValueError("Expected one combined FASTA and provenance table per sample")
if MIN_LENGTH < 1:
    raise ValueError("rna_min_contig_length must be positive")

## Assembly statistics

Every saved assembler FASTA has already been filtered at rna_min_contig_length before renaming and analysis. The combined FASTA additionally collapses exact duplicates and reverse complements. N50 is the contig length at which half the assembled bases are in contigs of at least that length.

In [ ]:
def fasta_lengths(path, include_ids=False):
    lengths = []
    identifiers = set()
    with open(path) as handle:
        for record in SeqIO.parse(handle, "fasta"):
            lengths.append(len(record.seq))
            if include_ids:
                if record.id in identifiers:
                    raise ValueError(f"Duplicate FASTA identifier in {path}: {record.id}")
                identifiers.add(record.id)
    return lengths, identifiers


def n50(lengths):
    if not lengths:
        return 0
    ordered = np.sort(np.asarray(lengths, dtype=np.int64))[::-1]
    return int(ordered[np.searchsorted(np.cumsum(ordered), ordered.sum() / 2)])


def assembly_statistics(sample, assembler, lengths):
    values = np.asarray(lengths, dtype=np.int64)
    return dict(sample=sample, assembler=assembler, contigs=len(values),
                total_bp=int(values.sum()), mean_bp=float(values.mean()) if len(values) else 0,
                median_bp=float(np.median(values)) if len(values) else 0,
                max_bp=int(values.max()) if len(values) else 0, N50_bp=n50(lengths),
                contigs_ge_500=int((values >= 500).sum()),
                contigs_ge_1000=int((values >= 1000).sum()),
                contigs_ge_10000=int((values >= 10000).sum()),
                contigs_ge_25000=int((values >= 25000).sum()),
                contigs_ge_50000=int((values >= 50000).sum()))


rows = []
lengths_by_sample = {}
combined_ids = {}
for sample_index, sample in enumerate(SAMPLES):
    for assembler_index, assembler in enumerate(ASSEMBLERS):
        path = assembly_files[sample_index * len(ASSEMBLERS) + assembler_index]
        lengths, _ = fasta_lengths(path)
        lengths_by_sample[(sample, assembler)] = lengths
        rows.append(assembly_statistics(sample, assembler, lengths))
    lengths, identifiers = fasta_lengths(combined_files[sample_index], include_ids=True)
    lengths_by_sample[(sample, "combined_derreplicated")] = lengths
    combined_ids[sample] = identifiers
    rows.append(assembly_statistics(sample, "combined_derreplicated", lengths))

summary_df = pd.DataFrame(rows)
summary_df.to_csv(snakemake.output.summary, index=False)
display(summary_df)

## QUAST statistics

QUAST assesses each selected RNA assembler and the combined FASTA separately, using unique sample/assembler labels. Every non-empty contig is included, without QUAST's default 500-bp cutoff. These are reference-free assembly statistics, not viral completeness or transcript-quality estimates. Empty assemblies are shown in the FASTA summary above and are skipped by QUAST. The standard DNA QUAST report is unchanged.

In [ ]:
quast_df = pd.read_csv(snakemake.input.quast, sep="\t")
quast_df

## Assembly reconciliation

A retained sequence appears once in the combined FASTA. An exact duplicate can originate from another assembler or the same assembler, including a reverse complement. Counts therefore depend on the order of the selected assemblers.

In [ ]:
statuses = ["retained", "exact_duplicate", "below_min_length"]
fate_rows = []
for sample_index, sample in enumerate(SAMPLES):
    provenance = pd.read_csv(provenance_files[sample_index], sep="\t")
    required = {"representative", "source", "contig_id", "length", "status"}
    if not required.issubset(provenance.columns):
        raise ValueError(f"Missing provenance columns for {sample}: {sorted(required - set(provenance.columns))}")
    unknown_sources = set(provenance["source"].dropna()) - set(ASSEMBLERS)
    unknown_statuses = set(provenance["status"].dropna()) - set(statuses)
    if unknown_sources or unknown_statuses or provenance["source"].isna().any() or provenance["status"].isna().any():
        raise ValueError(f"Unexpected source or status in the provenance table for {sample}")
    retained = provenance.loc[provenance["status"] == "retained", "representative"]
    if retained.isna().any() or set(retained) != combined_ids[sample] or len(retained) != len(combined_ids[sample]):
        raise ValueError(f"Combined FASTA identifiers do not match retained provenance for {sample}")
    if provenance.loc[provenance["status"] == "below_min_length", "length"].ge(MIN_LENGTH).any():
        raise ValueError(f"Below-minimum provenance entry reaches the configured length for {sample}")
    for assembler in ASSEMBLERS:
        subset = provenance.loc[provenance["source"] == assembler]
        if len(subset) != len(lengths_by_sample[(sample, assembler)]):
            raise ValueError(f"Assembly FASTA and provenance counts differ for {sample}/{assembler}")
        counts = subset["status"].value_counts()
        fate_rows.append(dict(sample=sample, assembler=assembler,
                              input_contigs=len(subset),
                              retained=int(counts.get("retained", 0)),
                              exact_duplicate=int(counts.get("exact_duplicate", 0)),
                              below_min_length=int(counts.get("below_min_length", 0))))

fate_df = pd.DataFrame(fate_rows)
fate_df.to_csv(snakemake.output.fate, index=False)
display(fate_df)

## Number of assembled contigs and assembled bases

Each sample shows every selected assembler and its combined assembly. The plots use raw counts and base totals, so zero-contig assemblies remain visible.

In [ ]:
plot_order = ASSEMBLERS + ["combined_derreplicated"]
palette = sns.color_palette("colorblind", len(plot_order))
figure_width = max(8, 0.9 * len(SAMPLES))
fig, ax = plt.subplots(figsize=(figure_width, 7))
summary_df.pivot(index="sample", columns="assembler", values="contigs").reindex(
    index=SAMPLES, columns=plot_order).plot.bar(ax=ax, color=palette, width=0.8)
ax.set_xlabel("Sample")
ax.set_ylabel("Number of contigs")
ax.legend(title="Assembly", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=90)
sns.despine(ax=ax)
fig.savefig(snakemake.output.contig_counts_png, bbox_inches="tight", transparent=True)
fig.savefig(snakemake.output.contig_counts_svg, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

fig, ax = plt.subplots(figsize=(figure_width, 7))
total_mbp = summary_df.assign(total_Mbp=summary_df["total_bp"] / 1_000_000)
total_mbp.pivot(index="sample", columns="assembler", values="total_Mbp").reindex(
    index=SAMPLES, columns=plot_order).plot.bar(ax=ax, color=palette, width=0.8)
ax.set_xlabel("Sample")
ax.set_ylabel("Total assembled length (Mbp)")
ax.legend(title="Assembly", bbox_to_anchor=(1.02, 1), loc="upper left")
ax.tick_params(axis="x", rotation=90)
sns.despine(ax=ax)
fig.savefig(snakemake.output.total_length_png, bbox_inches="tight", transparent=True)
fig.savefig(snakemake.output.total_length_svg, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

## Contig length distribution

The frequency curves pool contigs across samples for each assembler. Inspect the per-sample CSV for sample-level lengths and N50.

In [ ]:
fig, ax = plt.subplots(figsize=(10, 7))
all_lengths = [length for values in lengths_by_sample.values() for length in values if length > 0]
if all_lengths:
    maximum = max(all_lengths)
    bins = np.logspace(0, max(1, np.ceil(np.log10(maximum + 1))), 36)
    for assembler, color in zip(plot_order, palette):
        values = [length for sample in SAMPLES
                  for length in lengths_by_sample[(sample, assembler)] if length > 0]
        if values:
            counts, edges = np.histogram(values, bins=bins)
            ax.step(edges[:-1], counts, where="post", label=assembler, color=color, linewidth=2)
    ax.set_xscale("log")
    ax.legend(title="Assembly")
else:
    ax.text(0.5, 0.5, "No contigs assembled", ha="center", va="center", transform=ax.transAxes)
ax.set_xlabel("Contig length (bp)")
ax.set_ylabel("Number of contigs")
sns.despine(ax=ax)
fig.savefig(snakemake.output.length_distribution_png, bbox_inches="tight", transparent=True)
fig.savefig(snakemake.output.length_distribution_svg, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)

## Contribution to the combined assembly

Each bar partitions an assembler's saved contigs into retained sequences and exact duplicates (including reverse complements). The below-minimum column is retained for older provenance files; new assembler FASTAs are already length-filtered, so it is normally zero. The combined assembly is the union of retained sequences across assemblers, before VirSorter screening.

In [ ]:
labels = [f"{sample}\n{assembler}" for sample in SAMPLES for assembler in ASSEMBLERS]
figure_width = max(8, 0.7 * len(labels))
fig, ax = plt.subplots(figsize=(figure_width, 7))
positions = np.arange(len(fate_df))
bottom = np.zeros(len(fate_df), dtype=int)
for status, color in zip(statuses, sns.color_palette("colorblind", len(statuses))):
    counts = fate_df[status].to_numpy(dtype=int)
    ax.bar(positions, counts, bottom=bottom, label=status.replace("_", " "), color=color)
    bottom += counts
ax.set_xticks(positions)
ax.set_xticklabels(labels, rotation=90)
ax.set_xlabel("Sample and assembler")
ax.set_ylabel("Number of input contigs")
ax.legend(title="Combination status", bbox_to_anchor=(1.02, 1), loc="upper left")
sns.despine(ax=ax)
fig.savefig(snakemake.output.provenance_png, bbox_inches="tight", transparent=True)
fig.savefig(snakemake.output.provenance_svg, bbox_inches="tight", transparent=True)
plt.show()
plt.close(fig)